# 신용카드 사기 거래 탐지

## 프로젝트 개요
Kaggle의 Credit Card Fraud Detection 데이터셋을 활용하여 딥러닝 기반의 이진 분류 모델을 개발한다.  
데이터는 2013년 9월 유럽 신용카드 거래 데이터로, 284,807건 중 492건(약 0.17%)만 사기 거래이다.  
클래스 불균형(class imbalance)을 다루는 방법과 다양한 하이퍼파라미터 실험을 통해 최적 모델을 선정한다.

## 전체 파이프라인
1. 데이터 로드
2. EDA
3. 전처리 (스케일링 + 클래스 가중치)
4. 학습/검증/테스트 분할
5. 베이스 모델 정의
6. 하이퍼파라미터 실험 (5회)
7. 최종 모델 학습 및 저장
8. 테스트셋 평가 및 해석
9. 임계값 최적화

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense
from keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight

---
## 1. 데이터 로드
- `Time` &nbsp;&nbsp;&nbsp;: 거래로부터 경과 시간(초)
- `Amount`: 거래 금액
- `V1~V28`: PCA 변환된 익명 피처
- `Class` &nbsp;: 0 = 정상, 1 = 사기

In [2]:
DATA_PATH = 'data/creditcard.csv'
df = pd.read_csv(DATA_PATH)

print('=== 데이터 크기 (행, 열) ===')
print(df.shape)
print()

print('=== 상위 5행 미리보기 ===')
print(df.head())
print()

=== 데이터 크기 (행, 열) ===
(284807, 31)

=== 상위 5행 미리보기 ===
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  

---
## 2. EDA

신용카드 사기 탐지에서 클래스 불균형이 매우 심각하기 때문에 정확도(Accuracy)만으로 모델 성능을 평가할 수 없다.

In [3]:
# 클래스 불균형 시각화
class_counts = df['Class'].value_counts()
class_labels = ['정상 (0)', '사기 (1)']

print(f'클래스 불균형 비율 — 정상: {class_counts[0]:,} | 사기: {class_counts[1]:,}')
print(f'사기 거래 비율: {class_counts[1] / len(df) * 100:.4f}%')

클래스 불균형 비율 — 정상: 284,315 | 사기: 492
사기 거래 비율: 0.1727%


In [4]:
print('=== 전체 데이터 통계량 ===')
print(df.describe().T.to_string())

=== 전체 데이터 통계량 ===
           count          mean           std         min           25%           50%            75%            max
Time    284807.0  9.481386e+04  47488.145955    0.000000  54201.500000  84692.000000  139320.500000  172792.000000
V1      284807.0  1.175161e-15      1.958696  -56.407510     -0.920373      0.018109       1.315642       2.454930
V2      284807.0  3.384974e-16      1.651309  -72.715728     -0.598550      0.065486       0.803724      22.057729
V3      284807.0 -1.379537e-15      1.516255  -48.325589     -0.890365      0.179846       1.027196       9.382558
V4      284807.0  2.094852e-15      1.415869   -5.683171     -0.848640     -0.019847       0.743341      16.875344
V5      284807.0  1.021879e-15      1.380247 -113.743307     -0.691597     -0.054336       0.611926      34.801666
V6      284807.0  1.494498e-15      1.332271  -26.160506     -0.768296     -0.274187       0.398565      73.301626
V7      284807.0 -5.620335e-16      1.237094  -43.557242     

---
## 3. 전처리

**스케일링:**  
`Amount`(거래 금액)와 `Time`(경과 시간)은 다른 피처(`V1~V28`)와 스케일이 다르다.  
딥러닝 모델은 입력 값의 크기에 민감하므로, `MinMaxScaler`로 정규화한다.    
`V1~V28`은 이미 PCA 변환되어 표준화된 상태이므로 추가 스케일링이 불필요하다.

**클래스 가중치:**
정상 거래가 사기 거래보다 약 577배 많다.    
`compute_class_weight('balanced')`를 사용하면 소수 클래스(사기)에 더 높은 가중치를 부여하여
모델이 사기 거래를 더 중요하게 학습하도록 유도한다.

In [ ]:
# Amount와 Time에 MinMaxScaler 적용
df_processed = df.copy()

# 최솟값을 0, 최댓값을 1로 만들어 모든 데이터를 0과 1 사이의 값으로 압축
scaler = MinMaxScaler()

df_processed[['Amount', 'Time']] = scaler.fit_transform(df_processed[['Amount', 'Time']])

print('=== 스케일링 후 Amount, Time 범위 확인 ===')
print(f"Amount — min: {df_processed['Amount'].min():.4f}, max: {df_processed['Amount'].max():.4f}")
print(f"Time   — min: {df_processed['Time'].min():.4f}, max: {df_processed['Time'].max():.4f}")

=== 스케일링 후 Amount, Time 범위 확인 ===
Amount — min: 0.0000, max: 1.0000
Time   — min: 0.0000, max: 1.0000


In [6]:
# 피처(X)와 레이블(y) 분리
X = df_processed.drop(columns=['Class']).values
y = df_processed['Class'].values

print(f'X: {X.shape}')
print(f'y: {y.shape}')

X: (284807, 30)
y: (284807,)


In [ ]:
# 클래스 가중치 계산
# compute_class_weight 함수는 아래 공식으로 각 클래스의 가중치를 계산
# 전체 샘플 수 / (클래수 수 * 해당 클래스의 샘플 수)

# balanced는 클래스 빈도에 반비례하는 가중치를 계산
class_weights_array = compute_class_weight(class_weight = 'balanced', classes = np.unique(y), y = y)

# Keras가 요구하는 딕셔너리 형태로 변환
class_weight_dict = {
    0: class_weights_array[0],  # 정상 거래 가중치
    1: class_weights_array[1]   # 사기 거래 가중치
}

print('=== 클래스 가중치 ===')

print(f"정상 거래 (0): {class_weight_dict[0]:.4f}")
print(f"사기 거래 (1): {class_weight_dict[1]:.4f}")

print(f"\n가중치 비율 (사기/정상): {class_weight_dict[1]/class_weight_dict[0]:.1f}배")
print("사기 거래 1건의 손실이 정상 거래보다 훨씬 크게 반영됨")

=== 클래스 가중치 ===
정상 거래 (0): 0.5009
사기 거래 (1): 289.4380

가중치 비율 (사기/정상): 577.9배
사기 거래 1건의 손실이 정상 거래보다 훨씬 크게 반영됨


---
## 4. 학습 / 검증 / 테스트 분할

- **학습셋 (Train, 60%)**: 모델 파라미터를 업데이트하는 데 사용
- **검증셋 (Validation, 20%)**: 하이퍼파라미터 튜닝 및 EarlyStopping 기준으로 사용
- **테스트셋 (Test, 20%)**: 최종 성능 평가에만 사용

In [ ]:
# 전체 → 학습 + 검증(80%) / 테스트(20%)
# stratify = y로 레이블의 클래스 비율을 그대로 유지하면서 데이터 분리
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size = 0.2, stratify = y)

# 학습 + 검증 → 학습(75%) / 검증(25%)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size = 0.25, stratify = y_trainval)

print('=== 데이터 분할 결과 ===')

print(f'X_train: {X_train.shape} ({len(X_train) / len(X) * 100:.1f}%)')
print(f'X_val  : {X_val.shape}   ({len(X_val) / len(X) * 100:.1f}%)')
print(f'X_test : {X_test.shape}  ({len(X_test) / len(X) * 100:.1f}%)')
print()

# 각 셋의 클래스 비율 확인 (stratify 적용 확인)
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    fraud_pct = y_split.mean() * 100
    print(f'{name} 사기 비율: {fraud_pct:.4f}%')

=== 데이터 분할 결과 ===
X_train: (170883, 30) (60.0%)
X_val  : (56962, 30)   (20.0%)
X_test : (56962, 30)  (20.0%)

Train 사기 비율: 0.1726%
Val 사기 비율: 0.1738%
Test 사기 비율: 0.1720%


---
## 6. 하이퍼파라미터 실험

각 셀에서 하나의 값만 여러 후보로 바꾸고 나머지는 고정한다.  
이전 셀의 최적값이 다음 셀에 누적되어 최적 조합을 찾는다.

### 6-1. 활성화 함수 비교
고정: units=32, lr=0.001, dropout=0.3, batch=512

In [ ]:
act_results = []

for act in ['relu', 'tanh', 'sigmoid']:
    model = keras.Sequential()

    model.add(Dense(32, activation = act))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(32, activation = act))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = [tf.keras.metrics.AUC(name='auc')])

    # EarlyStopping 기준을 AUC 값을 기준으로 연속으로 5번 최댓값을 갱신하지 못 하면 학습을 멈춤
    model.fit(X_train, y_train, validation_data = (X_val, y_val),
              epochs = 100, batch_size = 512, class_weight = class_weight_dict,
              callbacks = [EarlyStopping(monitor = 'val_auc', patience = 5, restore_best_weights = True, mode = 'max', verbose = 0)],
              verbose = 0)

    loss, auc = model.evaluate(X_test, y_test)
    act_results.append({'activation': act, 'AUC': round(auc, 4)})

result_df = pd.DataFrame(act_results)
print(result_df.to_string())

# AUC 컬럼에서 최댓값을 가진 인덱스 번호를 행으로 하는 activation 컬럼 값 추출
best_activation = result_df.loc[result_df['AUC'].idxmax(), 'activation']
print(f'\n★ best activation: {best_activation}')

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 0s 209us/step - auc: 0.9942 - loss: 0.0673
1781/1781 ━━━━━━━━━━━━━━━━━━━━ 0s 201us/step - auc: 0.9931 - loss: 0.0856
1781/1781 ━━━━━━━━━━━━━━━━━━━━ 0s 201us/step - auc: 0.9935 - loss: 0.0948
  activation     AUC
0       relu  0.9942
1       tanh  0.9931
2    sigmoid  0.9935

★ best activation: relu


### 6-2. 학습률 비교
고정: best_activation, units=32, dropout=0.3, batch=512

In [10]:
lr_results = []

for lr in [0.01, 0.001, 0.0001, 0.00001]:
    model = keras.Sequential()

    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = lr), loss = 'binary_crossentropy', metrics = [tf.keras.metrics.AUC(name='auc')])
                           
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs = 100, batch_size = 512, class_weight = class_weight_dict,
              callbacks = [EarlyStopping(monitor = 'val_auc', patience = 5, restore_best_weights = True, mode = 'max', verbose = 0)],
              verbose = 0)
    
    loss, auc = model.evaluate(X_test, y_test, verbose = 0)
    lr_results.append({'lr': lr, 'AUC': round(auc, 4)})

result_df = pd.DataFrame(lr_results)
print(result_df.to_string(index = False))

best_lr = float(result_df.loc[result_df['AUC'].idxmax(), 'lr'])
print(f'\n★ best lr: {best_lr}')

     lr    AUC
0.01000 0.9909
0.00100 0.9924
0.00010 0.9911
0.00001 0.9875

★ best lr: 0.001


### 6-3. 배치 크기 비교
고정: best_activation, best_lr, dropout=0.3

In [11]:
bs_results = []

for bs in [256, 512, 1024, 2048]:
    model = keras.Sequential()

    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(0.3))
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = tf.keras.optimizers.Adam(best_lr), loss = 'binary_crossentropy', metrics = [tf.keras.metrics.AUC(name='auc')])
    
    model.fit(X_train, y_train, validation_data = (X_val, y_val),
              epochs = 100, batch_size = bs, class_weight = class_weight_dict,
              callbacks = [EarlyStopping(monitor='val_auc', patience = 5, restore_best_weights = True, mode = 'max', verbose = 0)],
              verbose = 0)
    
    loss, auc = model.evaluate(X_test, y_test, verbose = 0)
    bs_results.append({'batch_size': bs, 'AUC': round(auc, 4)})

result_df = pd.DataFrame(bs_results)
print(result_df.to_string(index = False))

best_batch = int(result_df.loc[result_df['AUC'].idxmax(), 'batch_size'])
print(f'\n★ best batch_size: {best_batch}')

 batch_size    AUC
        256 0.9920
        512 0.9840
       1024 0.9929
       2048 0.9942

★ best batch_size: 2048


### 6-4. 드롭아웃 비율 비교
고정: best_activation, best_lr, best_batch

In [12]:
do_results = []

for dr in [0.1, 0.2, 0.3, 0.5]:
    model = keras.Sequential()

    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(dr))
    model.add(Dense(32, activation = best_activation))
    model.add(tf.keras.layers.Dropout(dr))
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = tf.keras.optimizers.Adam(best_lr), loss = 'binary_crossentropy', metrics = [tf.keras.metrics.AUC(name='auc')])
    
    model.fit(X_train, y_train, validation_data = (X_val, y_val),
              epochs = 100, batch_size = best_batch, class_weight = class_weight_dict,
              callbacks = [EarlyStopping(monitor = 'val_auc', patience = 5, restore_best_weights = True, mode = 'max', verbose = 0)],
              verbose = 0)
    
    loss, auc = model.evaluate(X_test, y_test, verbose = 0)
    do_results.append({'dropout': dr, 'AUC': round(auc, 4)})

result_df = pd.DataFrame(do_results)
print(result_df.to_string(index = False))

best_dropout = float(result_df.loc[result_df['AUC'].idxmax(), 'dropout'])
print(f'\n★ best dropout: {best_dropout}')

 dropout    AUC
     0.1 0.9918
     0.2 0.9924
     0.3 0.9908
     0.5 0.9850

★ best dropout: 0.2


In [13]:
print('=== 최종 하이퍼파라미터 조합 ===')
print(f'  activation  : {best_activation}')
print(f'  learning_rate: {best_lr}')
print(f'  batch_size  : {best_batch}')
print(f'  dropout_rate: {best_dropout}')

=== 최종 하이퍼파라미터 조합 ===
  activation  : tanh
  learning_rate: 0.001
  batch_size  : 2048
  dropout_rate: 0.2


---
## 7. 최종 모델 학습 및 저장

각 실험에서 찾은 최적값 조합으로 **train + val** 전체를 사용해 재학습한 뒤 저장한다.

In [14]:
X_tv = np.concatenate([X_train, X_val])
y_tv = np.concatenate([y_train, y_val])

final_model = keras.Sequential()

final_model.add(Dense(32, activation = best_activation))
final_model.add(tf.keras.layers.Dropout(best_dropout))
final_model.add(Dense(32, activation = best_activation))
final_model.add(tf.keras.layers.Dropout(best_dropout))
final_model.add(Dense(1, activation = 'sigmoid'))

final_model.compile(optimizer = tf.keras.optimizers.Adam(best_lr), loss = 'binary_crossentropy', metrics = [tf.keras.metrics.AUC(name='auc')])

final_hist = final_model.fit(X_tv, y_tv, validation_data = (X_val, y_val), epochs = 100, batch_size = best_batch, class_weight = class_weight_dict,
                             callbacks = [EarlyStopping(monitor = 'val_auc', patience = 5, restore_best_weights = True, mode = 'max', verbose = 1)], verbose = 0)

final_model.save('best_fraud_model.keras')
print('모델 저장 완료: best_fraud_model.keras')

Epoch 71: early stopping
Restoring model weights from the end of the best epoch: 66.
모델 저장 완료: best_fraud_model.keras


---
## 8. 테스트셋 평가 및 해석  

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

loaded_model = tf.keras.models.load_model('best_fraud_model.keras')

loss, auc = loaded_model.evaluate(X_test, y_test)

y_pred_prob = loaded_model.predict(X_test).flatten()

# 0.5 임계값 기준으로 이상이면 사기, 미만이면 정상으로 이진 분류
y_pred_class = (y_pred_prob >= 0.5).astype(int)

auc_score = roc_auc_score(y_test, y_pred_prob)

print(f'loss    : {loss:.4f}')
print(f'AUC-ROC : {auc_score:.4f}')
print()

# Precision, Recall은 임계값 하나에서 계산
# Precision: 사기라고 예측한 것 중 얼마나 맞았나
# Recall: 실제 사기 중 얼마나 잡았나
print(classification_report(y_test, y_pred_class, target_names=['정상 (0)', '사기 (1)'], digits=4))

loss    : 0.0392
AUC-ROC : 0.9932

              precision    recall  f1-score   support

      정상 (0)     0.9999    0.9894    0.9946     56864
      사기 (1)     0.1313    0.9286    0.2301        98

    accuracy                         0.9893     56962
   macro avg     0.5656    0.9590    0.6124     56962
weighted avg     0.9984    0.9893    0.9933     56962



---
## 9. 임계값 최적화 (Threshold Optimization)

기본 임계값 0.5는 사기 거래 **재현율(Recall)** 을 극대화하지만, **정밀도(Precision)** 가 매우 낮아진다.  
검증셋에서 **F1-score가 최대**가 되는 임계값을 탐색한 뒤 테스트셋에 적용하여 균형 잡힌 성능을 확인한다.

In [ ]:
from sklearn.metrics import precision_recall_curve

# 검증셋 예측 확률 계산
y_val_prob = loaded_model.predict(X_val).flatten()

# 임계값을 바꿔가며 각 지점의 Precision, Recall을 배열로 반환
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)

# 각 임계값에서 F1-score 계산 후, F1이 최대가 되는 임계값 선택
# F1: Precision, Recall을 하나의 숫자로 합친 평균
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_threshold = thresholds[f1_scores.argmax()]

print(f'최적 임계값: {best_threshold:.4f}')

y_test_prob = loaded_model.predict(X_test).flatten()

# 테스트셋에 최적 임계값 적용
y_pred_opt  = (y_test_prob >= best_threshold).astype(int)

print()
print(classification_report(y_test, y_pred_opt, target_names=['정상 (0)', '사기 (1)'], digits=4))

최적 임계값: 0.9993

              precision    recall  f1-score   support

      정상 (0)     0.9996    0.9997    0.9996     56864
      사기 (1)     0.8152    0.7653    0.7895        98

    accuracy                         0.9993     56962
   macro avg     0.9074    0.8825    0.8946     56962
weighted avg     0.9993    0.9993    0.9993     56962

